# YOLOv5 数据质量修复审计

## tl;dr

- 清洗后共有 **1,511 / 188 / 188** 张训练、验证和测试图像。
- 三组之间的原始来源重叠和逐字节图像重复均为 **0**。
- 共移除 3,181 个重复或同来源多余版本，并把 26 个伪装成 JPEG 的 HEIC 文件转换为标准 JPEG。
- 旧模型指标仍只能作参考；可信的模型结论需要在清洗数据上重新训练。

## Context & Methods

目标是修复跨分组泄漏、重复图片、错误图片编码和不可直接运行的数据配置。拆分单位是 Roboflow 文件名中 `.rf.` 之前的原始来源名称；按来源所含类别组合分层，稳定分为 80%/10%/10%。训练集保留不同增强版本，验证和测试每个来源只保留一张代表图。

### Key Assumptions

- `.rf.` 之前的名称代表同一张原始图片。抽样图像支持这一假设。
- 空标签图是用于抑制误报的负样本。抽样中主要为笔、铅笔等香烟相似物。
- 旧训练结果没有在清洗后的测试集上重新计算。

## Data

加载清洗摘要和独立验证输出。

In [1]:
from pathlib import Path
import json
import pandas as pd

package_root = Path.cwd()
summary = json.loads((package_root / 'audit' / 'summary.json').read_text(encoding='utf-8'))
verification = json.loads((package_root / 'audit' / 'verification.json').read_text(encoding='utf-8'))
summary['generated_at'], verification['passed']

('2026-08-11T09:41:28.562610+00:00', True)

## Results

先比较清洗前后数据量，再核对泄漏和标签完整性。

In [2]:
rows = []
for split in ('train', 'valid', 'test'):
    before = summary['input']['split_counts'][split]
    after = summary['output'][split]
    rows.append({
        'split': split,
        'before_images': before['images'],
        'after_images': after['images'],
        'after_sources': after['sources'],
        'after_annotations': after['annotations'],
        'after_empty_labels': after['empty_labels'],
    })
pd.DataFrame(rows)

,split,before_images,after_images,after_sources,after_annotations,after_empty_labels
0,train,3871,1511,1511,1383,276
1,valid,881,188,188,171,34
2,test,316,188,188,171,34


In [3]:
pd.DataFrame([
    {'split_pair': pair, **counts}
    for pair, counts in verification['overlaps'].items()
]), verification['failures']

(    split_pair  source_names  exact_image_hashes  \
 0  train_valid             0                   0   
 1   train_test             0                   0   
 2   valid_test             0                   0   
 
    perceptual_hash_distance_le_4  
 0                              0  
 1                              0  
 2                              0  ,
 [])

In [4]:
pd.DataFrame([
    {
        'split': split,
        'cigarette': values.get('class_0', 0),
        'flame': values.get('class_1', 0),
        'smoke': values.get('class_2', 0),
        'boxes': values.get('box', 0),
        'polygons': values.get('polygon', 0),
    }
    for split, values in summary['output'].items()
])

,split,cigarette,flame,smoke,boxes,polygons
0,train,641,399,343,1257,126
1,valid,86,49,36,149,22
2,test,82,50,39,154,17


## Takeaways

- 数据层面的关键问题已经修复：配对完整、坐标合法、图片可解码、跨分组来源和哈希均隔离。
- 评估集按原始来源计数，而不是按增强图片计数，因此规模变小但可信度更高。
- 空标签负样本被保留；若标注规范并非有意加入相似物负样本，应再由标注负责人确认。
- 下一步应重新训练并单独运行 `val.py --task test`，旧的 mAP 不应继续作为最终结论。